In [1]:
import os
os.chdir("../")

In [2]:
%pwd

'd:\\Coding\\Development\\AI\\Medical Chatbot'

In [3]:
# %pip install langchain==0.3.26 flask==3.1.1 pypdf==5.6.1 sentence-transformers==4.1.0 python-dotenv==1.1.0 langchain-pinecone==0.2.8 langchain-openai==0.3.24 langchain-community==0.3.26

## Data Extraction

In [4]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [5]:
# Extract Text from PDF File
def load_pdf_files(data):
    loader = DirectoryLoader(data, glob="*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

In [6]:
extracted_data = load_pdf_files("data")

In [7]:
# extracted_data

In [8]:
len(extracted_data)

637

In [9]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of document objects, return a new list of document objects containing only 'source' in metdata and the original page_content
    """

    minimal_docs: List[Document] = []

    for doc in docs:
        src = doc.metadata.get("source")

        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )

    return minimal_docs

In [10]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [11]:
# minimal_docs

## Chunking (Split document into smaller chunks)

In [12]:
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [13]:
text_chunk = text_split(minimal_docs)
print(f"Number of Chunks: {len(text_chunk)}")

Number of Chunks: 5859


## Embedding Model

In [14]:
# %pip install --upgrade --force-reinstall numpy scikit-learn

In [15]:
# %pip install --force-reinstall "numpy==1.26.4" scikit-learn

In [16]:
# from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

def download_embedding():
    """
    Download and return the HuggingFace embeddings model 
    """

    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name = model_name
    )

    return embeddings

embedding = download_embedding()

C:\Users\DELL\AppData\Local\Temp\ipykernel_4960\971429451.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [17]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [18]:
vector = embedding.embed_query("Hello World")
# vector

In [19]:
print(f"Length of Vector: {len(vector)}")

Length of Vector: 384


## Pinecone Vector DB

In [20]:
from dotenv import load_dotenv
import os
load_dotenv()


True

In [21]:
MEDICALBOT_PINECONE_API_KEY = os.getenv("MEDICALBOT_PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

os.environ["MEDICALBOT_PINECONE_API_KEY"] = MEDICALBOT_PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [22]:
from pinecone import Pinecone
pinecone_api_key = MEDICALBOT_PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [23]:
pc

In [24]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(name=index_name, dimension=384, metric="cosine", spec=ServerlessSpec(cloud='aws', region="us-east-1"))

index = pc.Index(index_name)

In [26]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunk,
    embedding=embedding,
    index_name=index_name
)

Loading Existing Index

In [27]:
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

Add More Data to the existing Pinecone index

In [28]:
dummyDoc = Document(
    page_content="Hi this is Ninad More, a software engineer at Barclays",
    metadata={"source": "LinkedIn"}
)

In [29]:
docsearch.add_documents(documents=[dummyDoc])

['c113d439-2cd7-4f78-9ce0-4445d47b1da2']